# Orquestación: repartir el trabajo

El [cuaderno de MCP](herramientas-y-mcp.ipynb) terminó con un problema medido: al pasar de tres herramientas a doce, el acierto del agente se hundió de 8 sobre 9 a 3 sobre 9, y el coste fijo de las definiciones se triplicó.

El [capítulo](https://iraitzm.github.io/manual-ia-generativa/parts/agentes/orquestacion.html) propone la salida evidente: repartir el trabajo entre **varios agentes especializados**, cada uno con su prompt corto y su juego reducido de herramientas. Este cuaderno lo monta y comprueba si funciona.

La respuesta corta es que sí, pero mucho menos de lo que promete la intuición, y por una razón que el capítulo también anticipa. Eso es lo que hace interesante el ejercicio.

## Lo que vamos a medir

1. Un agente **monolítico** con las doce herramientas.
2. Un sistema **enrutado**: un clasificador barato decide, un especialista actúa.
3. El **techo** de ese sistema: qué pasaría si el enrutador nunca se equivocara.

La distancia entre los dos últimos es la lección del cuaderno.

## Preparación

In [ ]:
!pip install -q duckdb "transformers>=4.51" torch

In [ ]:
import pathlib
import subprocess
import sys

LOCAL = pathlib.Path("../../data/secretaria")
COLAB = pathlib.Path("manual-ia-generativa/data/secretaria")

if LOCAL.exists():
    base = LOCAL
else:
    if not COLAB.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--quiet",
             "https://github.com/IraitzM/manual-ia-generativa.git"],
            check=True,
        )
    base = COLAB

sys.path.insert(0, str(base.resolve()))

from secretaria import preparar

ctx = preparar()

In [ ]:
import json
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(ctx.modelo)
modelo = AutoModelForCausalLM.from_pretrained(ctx.modelo, dtype=torch.float32)
modelo.eval()

PATRON = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)
SISTEMA = ("Eres el asistente de la secretaría académica. Para responder debes llamar "
           "a una de las herramientas disponibles. No respondas de memoria.")


def esquema(nombre, descripcion, propiedades, obligatorios):
    return {"type": "function", "function": {
        "name": nombre, "description": descripcion,
        "parameters": {"type": "object", "properties": propiedades,
                       "required": obligatorios}}}

## Tres especialidades, cuatro herramientas cada una

Las doce herramientas del cuaderno anterior, repartidas en tres grupos por afinidad. No hemos añadido ni quitado nada: son exactamente las mismas descripciones.

In [ ]:
PLAZOS = [
    esquema("consultar_plazo", "Fechas de inicio y fin de un trámite administrativo.",
            {"tramite": {"type": "string",
                         "description": "beca, matricula, tfg, revision, grupo"}}, ["tramite"]),
    esquema("consultar_calendario", "Devuelve el calendario académico del curso.",
            {"curso": {"type": "string"}}, []),
    esquema("consultar_horario", "Horario de clases de una asignatura.",
            {"asignatura": {"type": "string"}}, ["asignatura"]),
    esquema("consultar_aula", "Aula asignada a una clase.",
            {"asignatura": {"type": "string"}}, ["asignatura"]),
]

EXPEDIENTE = [
    esquema("consultar_expediente", "Asignaturas y notas del alumno que pregunta.",
            {"asignatura": {"type": "string"}}, []),
    esquema("consultar_creditos", "Créditos superados y pendientes del alumno.", {}, []),
    esquema("consultar_convocatorias", "Convocatorias consumidas por el alumno.",
            {"asignatura": {"type": "string"}}, []),
    esquema("consultar_solicitudes", "Solicitudes abiertas del alumno.", {}, []),
]

NORMATIVA = [
    esquema("buscar_normativa", "Busca una regla o requisito en la normativa.",
            {"consulta": {"type": "string"}}, ["consulta"]),
    esquema("consultar_guia_docente", "Guía docente completa de una asignatura.",
            {"asignatura": {"type": "string"}}, ["asignatura"]),
    esquema("consultar_profesor", "Profesorado de una asignatura.",
            {"asignatura": {"type": "string"}}, ["asignatura"]),
    esquema("consultar_tasas", "Importe de las tasas de matrícula.",
            {"curso": {"type": "string"}}, []),
]

ESPECIALISTAS = {"plazo": PLAZOS, "expediente": EXPEDIENTE, "normativa": NORMATIVA}
TODAS = PLAZOS + EXPEDIENTE + NORMATIVA
CATEGORIAS = list(ESPECIALISTAS)

# (consulta, herramienta correcta, especialista al que debería ir)
CASOS = [
    ("¿hasta cuándo puedo pedir la beca?", "consultar_plazo", "plazo"),
    ("¿cuándo se abre la matrícula extraordinaria?", "consultar_plazo", "plazo"),
    ("¿qué día empiezan los exámenes de febrero?", "consultar_plazo", "plazo"),
    ("¿qué nota saqué en cálculo?", "consultar_expediente", "expediente"),
    ("¿de cuántas asignaturas estoy matriculado?", "consultar_expediente", "expediente"),
    ("¿qué notas tengo este curso?", "consultar_expediente", "expediente"),
    ("¿cuántas veces me puedo presentar a una asignatura?", "buscar_normativa", "normativa"),
    ("¿se puede convalidar experiencia laboral?", "buscar_normativa", "normativa"),
    ("¿qué requisitos piden para la beca general?", "buscar_normativa", "normativa"),
]

print(f"{len(TODAS)} herramientas repartidas en {len(ESPECIALISTAS)} especialistas")

## Las dos piezas

El **especialista** es el agente del cuaderno anterior, sin cambios, con su juego corto. El **enrutador** es el clasificador del [cuaderno de prompting](../contexto/prompting.ipynb): una sola pasada, decodificación restringida a las tres categorías.

Que el enrutador sea barato es parte del diseño. El capítulo lo dice: puede ser un modelo pequeño, porque decidir a quién pasar algo es más fácil que resolverlo.

In [ ]:
PRIMEROS = {c: tok(c, add_special_tokens=False).input_ids[0] for c in CATEGORIAS}
IDS = torch.tensor([PRIMEROS[c] for c in CATEGORIAS])

SISTEMA_ENRUTADOR = """Clasifica la consulta del alumno según a quién hay que pasarla:
- plazo: pregunta por una fecha, un plazo o un horario.
- expediente: pregunta por sus propios datos, notas o solicitudes.
- normativa: pregunta por una regla, un requisito o una guía docente.
Devuelve únicamente el identificador.

Ejemplos:
¿cuándo empieza el plazo de reconocimiento? -> plazo
¿tengo aprobada física? -> expediente
¿qué nota mínima piden para matrícula de honor? -> normativa
¿qué día es la defensa del trabajo? -> plazo
¿cuántos créditos llevo superados? -> expediente
¿se puede repetir un examen aprobado? -> normativa"""


def enrutar(consulta):
    """Decide el especialista. Una pasada, sin generar texto."""
    texto = tok.apply_chat_template(
        [{"role": "system", "content": SISTEMA_ENRUTADOR},
         {"role": "user", "content": consulta}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    entrada = tok(texto, return_tensors="pt")
    with torch.no_grad():
        logits = modelo(**entrada).logits[0, -1]
    probabilidades = torch.softmax(logits[IDS], dim=-1)
    indice = int(probabilidades.argmax())
    return CATEGORIAS[indice], int(entrada.input_ids.shape[1])


def elegir_herramienta(esquemas, consulta):
    """Un paso del especialista: qué herramienta pide y cuánto contexto gasta."""
    texto = tok.apply_chat_template(
        [{"role": "system", "content": SISTEMA}, {"role": "user", "content": consulta}],
        tools=esquemas, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    entrada = tok(texto, return_tensors="pt")
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=60, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
    bruto = tok.decode(salida[0][entrada.input_ids.shape[1]:], skip_special_tokens=True)
    encontrado = PATRON.search(bruto)
    nombre = None
    if encontrado:
        try:
            nombre = json.loads(encontrado.group(1)).get("name")
        except json.JSONDecodeError:
            pass
    return nombre, int(entrada.input_ids.shape[1])

## Las tres arquitecturas, medidas

In [ ]:
def monolitico():
    aciertos = coste = 0
    for consulta, esperada, _ in CASOS:
        nombre, tokens = elegir_herramienta(TODAS, consulta)
        aciertos += nombre == esperada
        coste += tokens
    return aciertos, coste


def enrutado():
    aciertos = coste = rutas_ok = 0
    for consulta, esperada, categoria_correcta in CASOS:
        categoria, t1 = enrutar(consulta)
        rutas_ok += categoria == categoria_correcta
        nombre, t2 = elegir_herramienta(ESPECIALISTAS[categoria], consulta)
        aciertos += nombre == esperada
        coste += t1 + t2
    return aciertos, coste, rutas_ok


def techo():
    """Lo que daría el sistema si el enrutador fuera perfecto."""
    aciertos = coste = 0
    for consulta, esperada, categoria_correcta in CASOS:
        nombre, tokens = elegir_herramienta(ESPECIALISTAS[categoria_correcta], consulta)
        aciertos += nombre == esperada
        coste += tokens
    return aciertos, coste


n = len(CASOS)
a_mono, c_mono = monolitico()
a_enr, c_enr, rutas = enrutado()
a_techo, c_techo = techo()

print(f"{'arquitectura':28s} {'acierto':>10s} {'tokens':>9s}")
print("-" * 50)
print(f"{'monolítico (12 herram.)':28s} {a_mono:>6d}/{n:<3d} {c_mono:>9d}")
print(f"{'enrutado':28s} {a_enr:>6d}/{n:<3d} {c_enr:>9d}")
print(f"{'techo (enrutado perfecto)':28s} {a_techo:>6d}/{n:<3d} {c_techo:>9d}")
print(f"\nEl enrutador acierta la categoría {rutas}/{n}")

Tres lecturas, y la tercera es la importante.

**El reparto ahorra contexto de verdad.** El sistema enrutado gasta bastantes menos tokens que el monolítico, y el de techo aún menos. Eso es lo que el capítulo llama **aislar**: cada especialista trabaja con su juego corto en lugar de arrastrar doce definiciones en cada vuelta. Y como el coste de un agente crece con el cuadrado de las vueltas, ese ahorro se multiplica.

**El reparto mejora el acierto, pero poco.** Un puñado de puntos sobre el monolítico. No es el salto que promete la intuición de "cada uno a lo suyo".

**Y el techo dice por qué.** Cuando cada consulta va a su especialista, el acierto sube bastante más. Toda esa diferencia entre el sistema real y su techo se la come **el enrutador**.

## El eslabón débil se lo lleva todo

El capítulo lo dice con aritmética: *"Cinco pasos al 90% de fiabilidad dan un 59%. La intuición de que entre varios se corrigen es falsa salvo que exista un verificador objetivo."*

Aquí solo hay dos pasos y ya se nota. Un sistema en cadena no promedia las fiabilidades de sus piezas: **las multiplica**.

In [ ]:
p_enrutador = rutas / n
p_especialista = a_techo / n

print(f"fiabilidad del enrutador:            {p_enrutador:.0%}")
print(f"fiabilidad del especialista:         {p_especialista:.0%}")
print(f"media de las dos:                    {(p_enrutador + p_especialista) / 2:.0%}")
print(f"producto (si fallaran independientes): {p_enrutador * p_especialista:.0%}")
print(f"MEDIDO de punta a punta:             {a_enr / n:.0%}")

print("\nY si encadenáramos más pasos independientes al 90%:")
for pasos in [1, 2, 3, 5, 10]:
    print(f"  {pasos:2d} pasos al 90%: {0.9 ** pasos:.0%}")

Aquí conviene parar, porque **el número medido no cuadra con el producto**: la multiplicación predice bastante menos de lo que sale.

No es un error de cuentas. Es que multiplicar supone que los dos pasos fallan de forma independiente, y no lo hacen. Las consultas fáciles lo son para las dos piezas y las difíciles también: cuando el enrutador acierta, suele ser porque la consulta era clara, y entonces el especialista también lo tiene fácil. Los aciertos van juntos.

Esa correlación es una buena noticia y conviene no sobrevenderla, porque **funciona igual en la otra dirección**: los fallos también se agrupan, y se concentran justo en los casos difíciles, que son los que justificaban el proyecto.

Lo que sí se sostiene, y es lo que importa al dibujar arquitecturas, es la forma de la curva. Un sistema en cadena **no promedia** las fiabilidades de sus piezas: no puede superar a la peor de ellas. Aquí el sistema entero se queda exactamente en el 44 % del enrutador, muy por debajo del 56 % que daría promediar y del 67 % del especialista.

La tabla del 90 % es el caso ideal con pasos independientes, y aun siendo optimista respecto a la correlación asusta: cinco pasos que funcionan el 90 % de las veces dan un sistema que funciona el 59 %.

Nada de esto significa que repartir sea mala idea. Significa que **el reparto solo compensa cuando cada paso es más fácil que el problema entero**. Enrutar es más fácil que resolver, así que la apuesta es razonable. Lo que no se puede es darla por ganada sin medir cada pieza por separado.

Y aquí la conclusión práctica es incómoda pero clara: antes de añadir más agentes, arreglad el enrutador. Es una sola pieza, es barata de evaluar y ahora mismo es el techo de todo lo demás.

## Los otros patrones

El enrutado es uno de los cinco que lista el capítulo. Los otros cuatro se montan igual de rápido y tienen sentidos muy distintos.

In [ ]:
def encadenado(consulta):
    """Salida de uno, entrada del siguiente. No es multiagente: es un flujo."""
    categoria, t1 = enrutar(consulta)
    nombre, t2 = elegir_herramienta(ESPECIALISTAS[categoria], consulta)
    return {"categoria": categoria, "herramienta": nombre, "tokens": t1 + t2}


def paralelo_votando(consulta, veces=3):
    """Varias opiniones sobre lo mismo y mayoría.

    Con `do_sample=False` las tres respuestas serían idénticas, así que aquí
    variamos el prompt en vez del muestreo: es la forma barata de conseguir
    diversidad real en lugar de repetir tres veces el mismo error.
    """
    import collections
    variantes = [SISTEMA_ENRUTADOR,
                 SISTEMA_ENRUTADOR.replace("Clasifica", "Decide y clasifica"),
                 SISTEMA_ENRUTADOR + "\nPiensa en qué dato hace falta para responder."]
    votos = []
    global SISTEMA_ENRUTADOR_ORIG
    for v in variantes[:veces]:
        texto = tok.apply_chat_template(
            [{"role": "system", "content": v}, {"role": "user", "content": consulta}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)
        entrada = tok(texto, return_tensors="pt")
        with torch.no_grad():
            logits = modelo(**entrada).logits[0, -1]
        votos.append(CATEGORIAS[int(torch.softmax(logits[IDS], dim=-1).argmax())])
    return collections.Counter(votos).most_common(1)[0][0], votos


consulta = "¿qué requisitos piden para la beca general?"
print("encadenado:", encadenado(consulta))
ganador, votos = paralelo_votando(consulta)
print(f"votación:   {ganador}  (votos: {votos})")

Mirad los votos. Los tres son iguales, y los tres están **mal**: la pregunta era por los requisitos de la beca, que es normativa.

Es la mejor demostración posible de para qué sirve y para qué no sirve votar. La votación ha costado tres llamadas y ha producido exactamente el mismo error que habría producido una, con una apariencia de consenso que además invita a fiarse más.

Votar solo ayuda si los votantes **fallan de formas distintas**. Tres variantes del mismo prompt sobre el mismo modelo no son tres opiniones: son la misma opinión repetida. Para conseguir diversidad real hace falta que cambie algo de fondo (modelos distintos, información distinta), y eso hay que comprobarlo antes de pagarlo, no suponerlo.

Los otros dos patrones del capítulo dan para más de lo que cabe aquí:

* **Orquestador y trabajadores.** El que la gente imagina al decir "multiagente". Un agente descompone, reparte y sintetiza. Es el enrutado con un paso más de agregación, y por tanto un factor más en la multiplicación de fiabilidades.
* **Evaluador y generador.** Uno produce, otro critica con criterios explícitos, el primero corrige. Es el único de los cinco que **puede** subir la fiabilidad en lugar de bajarla, y solo con una condición: que el criterio de calidad se pueda escribir. Cuando se puede (compila, pasa los tests, la fecha coincide con la tabla), funciona muy bien. Cuando no, es una discusión cara entre dos modelos que opinan.

Fijaos en que la condición del evaluador es la misma que el capítulo de agentes daba para saber si un caso de uso sale bien: **que el resultado se pueda verificar**.

## Lo que cuesta, en una tabla

El capítulo avisa de cuatro precios. Tres se pueden ver ya con lo que llevamos medido.

In [ ]:
print("1. TOKENS. Cada pieza arrastra su propio prompt de sistema.")
print(f"   enrutador solo: {c_enr - c_techo:5d} tokens de los {c_enr} del sistema")
print(f"   o sea, un {(c_enr - c_techo) / c_enr:.0%} del gasto se va en decidir, no en resolver.")

print("\n2. FIABILIDAD. Se multiplica, no se promedia.")
print(f"   {p_enrutador:.0%} x {p_especialista:.0%} = {p_enrutador * p_especialista:.0%}")

print("\n3. LATENCIA. Se acumula, salvo en ramas paralelas.")
print("   monolítico: 1 llamada al modelo por consulta")
print("   enrutado:   2 llamadas al modelo por consulta")

print("\n4. DEPURACIÓN. Sin trazas, arqueología. Es el cuaderno de observabilidad.")

Ese primer número merece un momento: una parte apreciable del presupuesto se gasta en **decidir quién trabaja**, no en trabajar. En este caso compensa porque el enrutador es diminuto y el ahorro de contexto en el especialista es mayor. Con un enrutador caro, la cuenta se da la vuelta.

De ahí la regla que cierra la sección en el capítulo, y que estos números respaldan: **un agente bien construido antes que tres mal repartidos**. Nosotros hemos repartido y hemos ganado poco, porque no arreglamos primero la pieza que decide.

## Ejercicios

**1. Arreglad el enrutador.** Es el techo de todo el sistema. Con lo aprendido en el cuaderno de prompting (ejemplos equilibrados, orden de las categorías, umbral de confianza), subid su acierto y volved a medir el sistema entero. ¿Cuánto del techo recuperáis?

**2. El enrutador que sabe dudar.** Haced que `enrutar` devuelva también la confianza y que, por debajo de un umbral, la consulta vaya al agente monolítico en lugar de a un especialista. Es lo mejor de los dos mundos, sobre el papel. Medid si lo es.

**3. Repartid de otra manera.** Los tres grupos los hemos hecho por afinidad temática. Probad a repartir por otro criterio (por ejemplo, lectura frente a escritura, o frecuencia de uso) y ved si el enrutador lo tiene más fácil.

**4. Evaluador y generador de verdad.** Montad un evaluador que compruebe una condición **objetiva**: que la fecha que da el agente coincida con la de `dim_plazo`. Cuando no coincida, devolvedle el error y dejadle corregirse. Medid cuántos casos arregla. Esta es la única forma de que encadenar suba la fiabilidad.

**5. ¿Y si no hiciera falta nada de esto?** Coged las nueve consultas y resolvedlas con las **tres** herramientas buenas del cuaderno anterior, sin las nueve de relleno ni enrutador. Comparad acierto y tokens con las tres arquitecturas de aquí. La pregunta del capítulo, con números: ¿el problema era la orquestación o el catálogo?

**6. La multiplicación en vuestro caso.** Escribid la fiabilidad que creéis que tiene cada paso de una arquitectura que hayáis dibujado alguna vez y multiplicadlas. Si el resultado os sorprende, ese es el ejercicio.

## Lo que os lleváis

* **Repartir ahorra contexto de verdad**, y ese ahorro se multiplica por el número de vueltas del bucle. Es la razón sólida para hacerlo.
* **Repartir no promedia la fiabilidad: la multiplica.** Dos pasos ya se notan; cinco al 90 % dejan el sistema en el 59 %.
* **Medid cada pieza por separado.** El sistema entero estaba limitado por el enrutador, y eso no se ve mirando solo el resultado final.
* **El enrutado solo compensa si decidir es más fácil que resolver.** Suele serlo, pero hay que comprobarlo.
* **Votar solo ayuda si los votantes fallan distinto.** Si no, es el mismo error a triple precio.
* **Evaluador y generador es el único patrón que puede subir la fiabilidad**, y solo si el criterio se puede escribir.
* Antes de añadir agentes, **arreglad el que ya tenéis**. Y antes de eso, mirad si el problema era el catálogo de herramientas.

Queda ver qué hacen los frameworks con todo esto: [la comparativa](frameworks.ipynb).